# Seminar 4: Alternating Least Squares (ALS)

## Goals

In this seminar we will:
1. Derive the **ALS optimization objective** and its **closed-form update rules**
2. Implement **ALS for explicit feedback** from scratch using analytical updates
3. Build **recommendation pipeline** on the **MovieLens 1M** dataset

In [ ]:
import os
import zipfile
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import sparse
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

plt.style.use("ggplot")

## 1. ALS formulation

We work with a **user–item rating matrix**:

- $R \in \mathbb{R}^{m \times n}$: $r_{ui}$ is the rating of user $u$ for item $i$ (only some entries are observed)
- $U \in \mathbb{R}^{m \times k}$: rows $u_u$ are **user latent factors**
- $V \in \mathbb{R}^{n \times k}$: rows $v_i$ are **item latent factors**
- $\Omega = \{(u,i): r_{ui} \text{ observed}\}$: set of known ratings

We want to approximate ratings as an inner product:

$$
\hat r_{ui} = u_u^\top v_i
$$

### Objective function (explicit ALS)

We minimize **squared error on observed entries** with L2 regularization:

$$
\min_{U,V} 
\sum_{(u,i) \in \Omega} (r_{ui} - u_u^\top v_i)^2 
+ \lambda \left( \sum_{u=1}^m \|u_u\|_2^2 + \sum_{i=1}^n \|v_i\|_2^2 \right),
$$
where $\lambda > 0$ is a regularization strength.

This problem is **not jointly convex** in $U, V$, but **is convex in one when the other is fixed**. ALS uses this fact.

### Alternating Least Squares idea

1. Fix **items** $V$, solve for all **users** $U$
2. Fix **users** $U$, solve for all **items** $V$
3. Repeat for several iterations

Each step decomposes into **independent ridge regressions** for every user (or item).

---

## 2. Closed-form update for one user

Fix the item matrix $V$. Consider user $u$ with observed ratings on a subset of items $I_u$.

Let:
- $r_u \in \mathbb{R}^{|I_u|}$: vector of ratings $r_{ui}$ for $i \in I_u$
- $V_u \in \mathbb{R}^{|I_u| \times k}$: matrix whose rows are $v_i^\top$ for items $i \in I_u$
- $u_u \in \mathbb{R}^k$: unknown user factor we want to find

The part of the loss that depends on $u_u$ is:

$$
L(u_u) = \sum_{i \in I_u} (r_{ui} - u_u^\top v_i)^2 + \lambda \|u_u\|_2^2.
$$

In matrix form:

$$
L(u_u) = \|r_u - V_u u_u\|_2^2 + \lambda \|u_u\|_2^2.
$$

This is a **ridge regression** problem with design matrix $V_u$, response $r_u$, and regularization $\lambda$.

Setting gradient to zero gives the **normal equations**:

$$
V_u^\top V_u \, u_u - V_u^\top r_u + \lambda u_u = 0.
$$

Re-arrange:

$$
(V_u^\top V_u + \lambda I_k) u_u = V_u^\top r_u.
$$

Thus the **closed-form update** is

$$
\boxed{u_u = (V_u^\top V_u + \lambda I_k)^{-1} V_u^\top r_u.}
$$

### Closed-form update for one item

By symmetry, when $U$ is fixed, each item factor $v_i$ solves:

$$
(U_i^\top U_i + \lambda I_k) v_i = U_i^\top r_i,
$$

where $U_i$ contains user factors for users who rated item $i$, and $r_i$ is the vector of their ratings. So:

$$
\boxed{v_i = (U_i^\top U_i + \lambda I_k)^{-1} U_i^\top r_i.}
$$

In practice we never form the matrix inverse explicitly; instead we solve the small $k \times k$ linear system with a solver like `np.linalg.solve`.

---

## 3. Explicit vs implicit ALS (short note)

In this seminar we implement **explicit ALS** for 1–5 star ratings.

There is also an **implicit-feedback ALS** variant (Hu, Koren, Volinsky, 2008) that works with binary interactions (views, clicks, plays). It uses a slightly different objective with **confidence weights** and treats unobserved entries as weak negatives. The update structure is still "alternating least squares", but the matrices in the normal equations change.

Here we focus on explicit ratings to keep the implementation clear.

## 4. Dataset: MovieLens 1M

We use the **MovieLens 1M** dataset:

- ~1,000,000 ratings on a 1–5 scale
- ~6,000 users and ~4,000 movies
- Each user has at least 20 ratings
- Data includes timestamps, which lets us do **chronological** train/test splits

The original data and description are available from the GroupLens website:
- https://grouplens.org/datasets/movielens/1m/

Format of the main ratings file `ratings.dat`:

```text
UserID::MovieID::Rating::Timestamp
```

In [ ]:
DATA_DIR = Path("ml-1m")
DATA_URL = "https://files.grouplens.org/datasets/movielens/ml-1m.zip"

if not DATA_DIR.exists():
    print("Downloading MovieLens 1M...")
    zip_path = Path("ml-1m.zip")
    urllib.request.urlretrieve(DATA_URL, zip_path)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(".")
    print("Done.")

ratings_path = DATA_DIR / "ratings.dat"
movies_path = DATA_DIR / "movies.dat"

ratings_cols = ["user_id", "item_id", "rating", "timestamp"]
ratings = pd.read_csv(
    ratings_path,
    sep="::",
    names=ratings_cols,
    engine="python",
    encoding="latin-1",
)
movies = pd.read_csv(
    movies_path,
    sep="::",
    names=["item_id", "title", "genres"],
    engine="python",
    encoding="latin-1",
)

print(f"Ratings: {len(ratings):,}")
print(f"Users:   {ratings['user_id'].nunique():,}")
print(f"Items:   {ratings['item_id'].nunique():,}")
ratings.head()

In [ ]:
n_users = ratings["user_id"].nunique()
n_items = ratings["item_id"].nunique()
sparsity = 1 - len(ratings) / (n_users * n_items)

print(f"Number of ratings: {len(ratings):,}")
print(f"Unique users:      {n_users:,}")
print(f"Unique items:      {n_items:,}")
print(f"Sparsity:          {sparsity:.4f} ({sparsity*100:.2f}%)")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(ratings["rating"], bins=5, edgecolor="black", alpha=0.7)
axes[0].set_xlabel("Rating")
axes[0].set_ylabel("Count")
axes[0].set_title("Rating distribution")

user_counts = ratings.groupby("user_id").size()
axes[1].hist(user_counts, bins=50, edgecolor="black", alpha=0.7)
axes[1].set_xlabel("Number of ratings per user")
axes[1].set_ylabel("Number of users")
axes[1].set_yscale("log")
axes[1].set_title("Ratings per user (log scale)")

plt.tight_layout()
plt.show()

In [ ]:
user_ids = ratings["user_id"].unique()
item_ids = ratings["item_id"].unique()

user_id_to_idx = {uid: idx for idx, uid in enumerate(user_ids)}
item_id_to_idx = {iid: idx for idx, iid in enumerate(item_ids)}

ratings["user_idx"] = ratings["user_id"].map(user_id_to_idx)
ratings["item_idx"] = ratings["item_id"].map(item_id_to_idx)

n_users = len(user_ids)
n_items = len(item_ids)

rows = ratings["user_idx"].to_numpy()
cols = ratings["item_idx"].to_numpy()
vals = ratings["rating"].to_numpy().astype(np.float32)

R_csr = sparse.csr_matrix((vals, (rows, cols)), shape=(n_users, n_items))

print(R_csr.shape, R_csr.nnz)

## 5. ALS implementation (explicit ratings)

We now implement ALS for explicit ratings using the closed-form updates derived above.

Implementation outline:

- Precompute **CSR** (user-by-item) and **CSC** (item-by-user) views of the rating matrix
- Alternate for `n_iters` iterations:
  - For each user, solve $(V_u^\top V_u + \lambda I) u_u = V_u^\top r_u$
  - For each item, solve $(U_i^\top U_i + \lambda I) v_i = U_i^\top r_i$
- Store learned user and item embeddings in matrices `U` and `V`.

$ R \in R^{n x k}, U \in R^{n x d}, V \in R^{d x k} $ 

In [ ]:
class ALSExplicit:
    def __init__(self, n_factors=40, n_iters=10, reg=0.1, random_state=42):
        self.n_factors = n_factors
        self.n_iters = n_iters
        self.reg = reg
        self.random_state = random_state

        self.user_factors = None  # shape (n_users, k)
        self.item_factors = None  # shape (n_items, k)

    def _init_factors(self, n_users, n_items):
        rng = np.random.default_rng(self.random_state)
        self.user_factors = 0.01 * rng.standard_normal(size=(n_users, self.n_factors))
        self.item_factors = 0.01 * rng.standard_normal(size=(n_items, self.n_factors))

    def fit(self, R_csr):
        """Fit ALS model on a CSR matrix of shape (n_users, n_items).

        R_csr is assumed to contain explicit ratings (zeros = missing).
        """
        if not sparse.isspmatrix_csr(R_csr):
            raise ValueError("R_csr must be a scipy.sparse.csr_matrix")

        n_users, n_items = R_csr.shape
        self._init_factors(n_users, n_items)

        # Precompute CSC for efficient per-item access
        R_csc = R_csr.tocsc()

        I_k = np.eye(self.n_factors, dtype=np.float32)

        for it in range(self.n_iters):
            # Update user factors U
            for u in tqdm(range(n_users), desc=f"ALS iter {it + 1}/{self.n_iters} [users]", leave=False):
                start, end = R_csr.indptr[u], R_csr.indptr[u + 1]
                item_idx = R_csr.indices[start:end]
                if item_idx.size == 0:
                    continue
                ratings_u = R_csr.data[start:end]

                V_u = self.item_factors[item_idx]  # shape (n_items_u, k)
                A = V_u.T @ V_u + self.reg * I_k
                b = V_u.T @ ratings_u

                self.user_factors[u] = np.linalg.solve(A, b)

            # Update item factors V
            for i in tqdm(range(n_items), desc=f"ALS iter {it + 1}/{self.n_iters} [items]", leave=False):
                start, end = R_csc.indptr[i], R_csc.indptr[i + 1]
                user_idx = R_csc.indices[start:end]
                if user_idx.size == 0:
                    continue
                ratings_i = R_csc.data[start:end]

                U_i = self.user_factors[user_idx]  # shape (n_users_i, k)
                A = U_i.T @ U_i + self.reg * I_k
                b = U_i.T @ ratings_i

                self.item_factors[i] = np.linalg.solve(A, b)

        return self

    def predict_for_user(self, user_idx):
        """Predict scores for all items for a given user index."""
        if self.user_factors is None or self.item_factors is None:
            raise ValueError("Model is not fitted yet")
        return self.item_factors @ self.user_factors[user_idx]

    def predict_single(self, user_idx, item_idx):
        """Predict a single rating r_hat[u, i]."""
        return float(self.user_factors[user_idx] @ self.item_factors[item_idx])

## 6. Sanity checks on ALS implementation

We start with basic shape checks and a small synthetic example where we know the ground-truth factors.

In [ ]:
n_users_small = 20
n_items_small = 30

rng = np.random.default_rng(0)
rows_small = rng.integers(0, n_users_small, size=200)
cols_small = rng.integers(0, n_items_small, size=200)
vals_small = rng.integers(1, 6, size=200).astype(np.float32)

R_small = sparse.csr_matrix((vals_small, (rows_small, cols_small)), shape=(n_users_small, n_items_small))

als_small = ALSExplicit(n_factors=8, n_iters=3, reg=0.1, random_state=0)
als_small.fit(R_small)

U = als_small.user_factors
V = als_small.item_factors

print("U shape:", U.shape)
print("V shape:", V.shape)

assert U.shape == (n_users_small, 8)
assert V.shape == (n_items_small, 8)

scores = als_small.predict_for_user(0)
print("Scores shape:", scores.shape)
assert scores.shape == (n_items_small,)
assert np.isfinite(scores).all()

In [ ]:
# Synthetic low-rank example: R = U_true V_true^T with missing entries

n_users_syn = 30
n_items_syn = 40
k_syn = 5

rng = np.random.default_rng(1)
U_true = rng.standard_normal(size=(n_users_syn, k_syn)).astype(np.float32)
V_true = rng.standard_normal(size=(n_items_syn, k_syn)).astype(np.float32)

R_full = U_true @ V_true.T

# Keep only a random subset of entries as "observed" ratings
mask = rng.random(size=R_full.shape) < 0.3
rows_syn, cols_syn = np.where(mask)
vals_syn = R_full[rows_syn, cols_syn].astype(np.float32)

R_syn = sparse.csr_matrix((vals_syn, (rows_syn, cols_syn)), shape=(n_users_syn, n_items_syn))

als_syn = ALSExplicit(n_factors=k_syn, n_iters=10, reg=0.01, random_state=1)
als_syn.fit(R_syn)

R_hat = als_syn.user_factors @ als_syn.item_factors.T

# Compute mean squared error on observed entries
mse_observed = np.mean((R_hat[rows_syn, cols_syn] - vals_syn) ** 2)
print(f"MSE on observed entries: {mse_observed:.4f}")

## 7. Train/test split and simple evaluation

We use a minimal **leave-last-out** strategy:

- Keep only users with at least `MIN_RATINGS` interactions
- For each such user, put the **last** rating (by timestamp) into the **test** set
- All earlier ratings go into the **train** set

We then train ALS on the train matrix and evaluate with **HitRate@K** and **MRR@K**:

- **HitRate@K**: fraction of users whose held-out item appears in top-K recommendations
- **MRR@K** (Mean Reciprocal Rank): average of 1/rank, where rank is the position of the held-out item in top-K (0 if not found)

In [ ]:
# Leave-last-out split

MIN_RATINGS = 5

user_counts = ratings.groupby("user_idx").size()
active_users = user_counts[user_counts >= MIN_RATINGS].index.to_numpy()

ratings_active = ratings[ratings["user_idx"].isin(active_users)].copy()

ratings_active = ratings_active.sort_values(["user_idx", "timestamp"])

# Mark the last interaction per user as test
ratings_active["rank"] = ratings_active.groupby("user_idx")["timestamp"].rank(method="first", ascending=False)

test_df = ratings_active[ratings_active["rank"] == 1].copy()
train_df = ratings_active[ratings_active["rank"] > 1].copy()

print(f"Train ratings: {len(train_df):,}")
print(f"Test ratings:  {len(test_df):,}")
print(f"Users in test: {test_df['user_idx'].nunique():,}")

# Build train CSR matrix
train_rows = train_df["user_idx"].to_numpy()
train_cols = train_df["item_idx"].to_numpy()
train_vals = train_df["rating"].to_numpy().astype(np.float32)

R_train = sparse.csr_matrix((train_vals, (train_rows, train_cols)), shape=(n_users, n_items))

# Ground-truth held-out item per user
heldout = dict(zip(test_df["user_idx"].to_numpy(), test_df["item_idx"].to_numpy()))

len(heldout)

In [ ]:
def hit_rate_at_k_als(model, R_train, heldout, k=10):
    """Compute HitRate@K for ALS model.

    model: fitted ALSExplicit
    R_train: CSR matrix with training interactions
    heldout: dict[user_idx -> item_idx] with single held-out item per user
    """
    n_users, n_items = R_train.shape
    hits = 0
    total = 0

    for u, item_true in heldout.items():
        # Items seen in training for this user
        start, end = R_train.indptr[u], R_train.indptr[u + 1]
        seen_items = set(R_train.indices[start:end])

        scores = model.predict_for_user(u)

        # Filter out seen items by setting their score to -inf
        scores_filtered = scores.copy()
        if seen_items:
            scores_filtered[list(seen_items)] = -np.inf

        top_k = np.argpartition(-scores_filtered, k)[:k]

        if item_true in top_k:
            hits += 1
        total += 1

    return hits / total if total > 0 else 0.0

In [ ]:
def mrr_at_k(model, R_train, heldout, k=10):
    """Compute Mean Reciprocal Rank@K for model with predict_for_user."""
    total = 0
    rr_sum = 0.0
    for u, item_true in heldout.items():
        start, end = R_train.indptr[u], R_train.indptr[u + 1]
        seen_items = set(R_train.indices[start:end])
        scores = model.predict_for_user(u)
        scores_filtered = scores.copy()
        if seen_items:
            scores_filtered[list(seen_items)] = -np.inf
        top_k = np.argpartition(-scores_filtered, k)[:k]
        top_k = top_k[np.argsort(-scores_filtered[top_k])]
        try:
            rank = np.where(top_k == item_true)[0][0] + 1
            rr_sum += 1.0 / rank
        except IndexError:
            pass
        total += 1
    return rr_sum / total if total > 0 else 0.0


def mrr_from_recs(recs, heldout, k=10):
    """Compute MRR@K from recs dict: recs[user_idx] -> list of item_idx."""
    total = 0
    rr_sum = 0.0
    for u, item_true in heldout.items():
        top_k = recs.get(u, [])[:k]
        try:
            rank = top_k.index(item_true) + 1
            rr_sum += 1.0 / rank
        except ValueError:
            pass
        total += 1
    return rr_sum / total if total > 0 else 0.0

In [ ]:
als_movielens = ALSExplicit(
    n_factors=40, 
    n_iters=10, 
    reg=0.1
)

als_movielens.fit(R_train)

K = 10
hr = hit_rate_at_k_als(als_movielens, R_train, heldout, k=K)
mrr = mrr_at_k(als_movielens, R_train, heldout, k=K)
print(f"HitRate@{K}: {hr:.4f}, MRR@{K}: {mrr:.4f}")

## 8. Hyperparameter tuning with Optuna

We use [Optuna](https://optuna.org/) to tune ALS hyperparameters: `n_factors`, `n_iters`, and `reg`. The objective is to maximize **HitRate@10** on the held-out test set. We run at most 10 trials to keep runtime reasonable.

In [ ]:
!pip install optuna

In [ ]:
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)


def objective(trial):
    n_factors = trial.suggest_int("n_factors", 20, 80, step=20)
    n_iters = trial.suggest_int("n_iters", 5, 15, step=5)
    reg = trial.suggest_float("reg", 0.01, 1.0, log=True)

    model = ALSExplicit(n_factors=n_factors, n_iters=n_iters, reg=reg, random_state=42)
    model.fit(R_train)

    hr = hit_rate_at_k_als(model, R_train, heldout, k=10)
    return hr


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=10, show_progress_bar=True)

print("Best HitRate@10:", study.best_value)
print("Best params:", study.best_params)

## 9. Comparison with library algorithms

We compare ALS with two popular library-based recommenders on the same MovieLens 1M setup:

1. **Item-based KNN** (from `implicit`): Uses cosine similarity between items; scores items by weighted sum of similar items the user has rated.
2. **EASE** (from `rectools`): Embarrassingly Shallow Autoencoders (Steck 2019) — a linear item-item model with closed-form solution.

In [ ]:
!pip install implicit rectools

In [ ]:
from implicit.nearest_neighbours import CosineRecommender

# Item-based KNN: fit on user-item matrix, computes item-item cosine similarity
# For explicit ratings we use the rating values (implicit treats them as interaction strengths)
class ItemKNNWrapper:
    """Wrapper to provide predict_for_user interface for hit_rate_at_k_als."""

    def __init__(self, K=50):
        self.model = CosineRecommender(K=K)
        self._scores = None

    def fit(self, R):
        R_f32 = R.astype(np.float32)
        self.model.fit(R_f32)
        # Scores: (user, item) = sum over j of R[u,j] * sim(j, i)
        self._scores = R_f32 @ self.model.similarity
        return self

    def predict_for_user(self, user_idx):
        row = self._scores[user_idx]
        return np.asarray(row.toarray()).flatten()

i2i_wrapper = ItemKNNWrapper(K=10)
i2i_wrapper.fit(R_train)

hr_i2i = hit_rate_at_k_als(i2i_wrapper, R_train, heldout, k=K)
mrr_i2i = mrr_at_k(i2i_wrapper, R_train, heldout, k=K)
print(f"Item-based KNN HitRate@{K}: {hr_i2i:.4f}, MRR@{K}: {mrr_i2i:.4f}")

In [ ]:
from rectools import Columns
from rectools.dataset import Dataset as RTDataset
from rectools.models.ease import EASEModel

# Prepare data for rectools (requires User, Item, Weight, Datetime)
rt_train = train_df[["user_idx", "item_idx", "rating", "timestamp"]].copy()
rt_train = rt_train.rename(columns={
    "user_idx": Columns.User,
    "item_idx": Columns.Item,
    "rating": Columns.Weight,
    "timestamp": Columns.Datetime,
})
rt_train[Columns.Datetime] = pd.to_datetime(rt_train[Columns.Datetime], unit="s")

rt_dataset = RTDataset.construct(rt_train)

ease_model = EASEModel(regularization=1000.0)
ease_model.fit(rt_dataset)

# Evaluate: recommend for each test user and compute HitRate@K
eval_users = list(heldout.keys())
ease_recs_df = ease_model.recommend(
    users=eval_users,
    dataset=rt_dataset,
    k=K,
    filter_viewed=True,
)

recs_ease = {}
for uid, group in ease_recs_df.groupby(Columns.User):
    recs_ease[int(uid)] = [int(i) for i in group[Columns.Item].tolist()]

hits_ease = sum(1 for u, item_true in heldout.items() if item_true in recs_ease.get(u, [])[:K])
hr_ease = hits_ease / len(heldout) if heldout else 0.0
mrr_ease = mrr_from_recs(recs_ease, heldout, k=K)
print(f"EASE HitRate@{K}: {hr_ease:.4f}, MRR@{K}: {mrr_ease:.4f}")

In [ ]:
# Summary comparison (ALS uses best params from Optuna)
als_best = ALSExplicit(**study.best_params, random_state=42)
als_best.fit(R_train)
hr_als = hit_rate_at_k_als(als_best, R_train, heldout, k=K)
mrr_als = mrr_at_k(als_best, R_train, heldout, k=K)

comparison = pd.DataFrame({
    "Model": ["ALS (from scratch)", "Item-based KNN (implicit)", "EASE (rectools)"],
    f"HitRate@{K}": [hr_als, hr_i2i, hr_ease],
    f"MRR@{K}": [mrr_als, mrr_i2i, mrr_ease],
})
print(comparison.to_string(index=False))

## 10. Recommendation pipeline with ALS

We now build helper functions that, given a raw `user_id`,

- map it to an internal `user_idx`
- score all items with the trained ALS model
- filter out items seen in training
- return top-N recommendations with movie titles.

In [ ]:
item_idx_to_id = {idx: iid for iid, idx in item_id_to_idx.items()}
item_id_to_title = dict(zip(movies["item_id"], movies["title"]))


def recommend_for_user(raw_user_id, model, R_train, top_n=10):
    """Return top-N (item_id, title, score) recommendations for a raw user_id."""
    if raw_user_id not in user_id_to_idx:
        raise ValueError(f"Unknown user_id: {raw_user_id}")

    user_idx = user_id_to_idx[raw_user_id]

    # Items seen in training for this user
    start, end = R_train.indptr[user_idx], R_train.indptr[user_idx + 1]
    seen_items = set(R_train.indices[start:end])

    scores = model.predict_for_user(user_idx)
    scores_filtered = scores.copy()
    if seen_items:
        scores_filtered[list(seen_items)] = -np.inf

    top_idx = np.argpartition(-scores_filtered, top_n)[:top_n]
    top_idx = top_idx[np.argsort(-scores_filtered[top_idx])]

    recs = []
    for idx in top_idx:
        item_id = item_idx_to_id[idx]
        title = item_id_to_title.get(item_id, "<unknown>")
        recs.append((int(item_id), title, float(scores_filtered[idx])))

    return recs

In [ ]:
example_users = list(heldout.keys())[:3]

for u_idx in example_users:
    raw_uid = user_ids[u_idx]
    print("=" * 80)
    print(f"User {raw_uid} (internal idx {u_idx})")

    # Show last few movies they rated highly in train
    user_hist = train_df[train_df["user_idx"] == u_idx]
    user_hist = user_hist.sort_values("timestamp", ascending=False).head(5)
    print("Recent train interactions:")
    for _, row in user_hist.iterrows():
        mid = row["item_id"]
        title = item_id_to_title.get(mid, "<unknown>")
        print(f"  {title} (rating={row['rating']})")

    print("\nTop-10 ALS recommendations:")
    recs = recommend_for_user(raw_uid, als_movielens, R_train, top_n=10)
    for item_id, title, score in recs:
        print(f"  {title}  (score={score:.3f})")

## 11. Summary

In this seminar we:

- Formulated matrix factorization with **ALS** as minimizing squared error on observed ratings with L2 regularization, and derived the **closed-form normal-equation updates** for user and item factors.
- Implemented an **explicit-feedback ALS** algorithm from scratch.
- Built a simple **leave-last-out evaluation** on MovieLens 1M with **HitRate@K** and **MRR@K** metrics, and a practical **recommendation pipeline** that returns top-N movie suggestions for a given user.